In [1]:
import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData

def load_and_segment(file_path, min_points=100):
    plydata = PlyData.read(file_path)
    data = plydata.elements[0].data
    
    points = np.vstack([data['x'], data['y'], data['z']]).T
    labels = data['scalar_Label']
    
    unique_labels = np.unique(labels)
    segments = {}
    
    for label in unique_labels:
        mask = labels == label
        segment_points = points[mask]
        
        if len(segment_points) >= min_points:
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(segment_points)
            segments[label] = pcd
            
    return segments

def normalize_pcd(pcd):
    points = np.asarray(pcd.points)
    centroid = np.mean(points, axis=0)
    points = points - centroid
    max_dist = np.max(np.sqrt(np.sum(points**2, axis=1)))
    
    if max_dist > 0:
        points = points / max_dist
        
    pcd.points = o3d.utility.Vector3dVector(points)
    return pcd, centroid, max_dist

def restore_mesh_scale(mesh, centroid, max_dist):
    vertices = np.asarray(mesh.vertices)
    if max_dist > 0:
        vertices = vertices * max_dist
    vertices = vertices + centroid
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    return mesh

def analyze_geometry(pcd):
    points = np.asarray(pcd.points)
    centroid = np.mean(points, axis=0)
    centered = points - centroid
    cov_matrix = np.cov(centered.T)
    eigenvalues, _ = np.linalg.eigh(cov_matrix)
    
    eigenvalues = np.sort(eigenvalues)[::-1]
    total_var = np.sum(eigenvalues)
    
    if total_var == 0:
        return 'unknown'
        
    e1, e2, e3 = eigenvalues / total_var
    
    if e3 < 0.05 and e1 > 0.4 and e2 > 0.4:
        return 'flat'
    elif e1 > 0.6 and e2 < 0.2 and e3 < 0.2:
        return 'tubular'
    elif e1 < 0.45 and e2 < 0.45 and e3 > 0.1:
        return 'spherical'
    else:
        return 'complex'

def reconstruct_segment(pcd, geom_type):
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    pcd.orient_normals_consistent_tangent_plane(100)
    
    if geom_type == 'flat':
        distances = pcd.compute_nearest_neighbor_distance()
        avg_dist = np.mean(np.asarray(distances))
        alpha = avg_dist * 3.0 
        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    elif geom_type in ['tubular', 'spherical']:
        radii = [0.02, 0.05, 0.1, 0.2]
        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
            pcd, o3d.utility.DoubleVector(radii))
    else:
        mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8, linear_fit=True)
        vertices_to_remove = densities < np.quantile(densities, 0.05)
        mesh.remove_vertices_by_mask(vertices_to_remove)
        
    mesh.compute_vertex_normals()
    return mesh

def evaluate_quality(pcd, mesh):
    if len(mesh.vertices) == 0:
        return float('inf')
    
    sampled_pcd = mesh.sample_points_uniformly(number_of_points=len(pcd.points))
    dists = pcd.compute_point_cloud_distance(sampled_pcd)
    dists = np.asarray(dists)
    rmse = np.sqrt(np.mean(dists**2))
    return rmse

def process_single_file(input_file, output_dir):
    filename = os.path.basename(input_file)
    output_file = os.path.join(output_dir, f"reconstructed_{filename}")
    
    segments = load_and_segment(input_file)
    final_mesh = o3d.geometry.TriangleMesh()
    
    total_rmse = 0.0
    valid_segments = 0
    
    for label, pcd in segments.items():
        cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
        pcd_clean = pcd.select_by_index(ind)
        
        if len(pcd_clean.points) < 50:
            continue
            
        pcd_norm, centroid, max_dist = normalize_pcd(pcd_clean)
        geom_type = analyze_geometry(pcd_norm)
        
        try:
            mesh = reconstruct_segment(pcd_norm, geom_type)
            
            if len(mesh.vertices) > 0:
                rmse = evaluate_quality(pcd_norm, mesh)
                total_rmse += rmse
                valid_segments += 1
                
                mesh = restore_mesh_scale(mesh, centroid, max_dist)
                final_mesh += mesh
        except Exception:
            continue
            
    if valid_segments > 0:
        avg_rmse = total_rmse / valid_segments
        print(f"[{filename}] Сборка завершена  Сегментов: {valid_segments} Отклонение (RMSE): {avg_rmse:.6f}")
    else:
        print(f"[{filename}] Ошибка сборки модели.")

    final_mesh.remove_duplicated_vertices()
    final_mesh.remove_duplicated_triangles()
    final_mesh.remove_degenerate_triangles()
    
    o3d.io.write_triangle_mesh(output_file, final_mesh)

def starts():
    input_dir = 'dataset'
    output_dir = 'output_meshes'
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    all_files = glob.glob(os.path.join(input_dir, '*.ply'))

        
    for file_path in sorted(all_files):
        process_single_file(file_path, output_dir)
starts()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[valve_0001_lidar_classes.ply] Сборка завершена  Сегментов: 5 Отклонение (RMSE): 0.055888
[valve_0002_lidar_classes.ply] Сборка завершена  Сегментов: 6 Отклонение (RMSE): 0.057205
[valve_0003_lidar_classes.ply] Сборка завершена  Сегментов: 7 Отклонение (RMSE): 0.071894
[valve_0004_lidar_classes.ply] Сборка завершена  Сегментов: 6 Отклонение (RMSE): 0.064501
[valve_0005_lidar_classes.ply] Сборка завершена  Сегментов: 7 Отклонение (RMSE): 0.086384
[valve_0006_lidar_classes.ply] Сборка завершена  Сегментов: 7 Отклонение (RMSE): 0.075839
[valve_0007_lidar_classes.ply] Сборка завершена  Сегментов: 5 Отклонение (RMSE): 0.050946
[valve_0008_lidar_classes.ply] Сборка завершена  Сегментов: 7 Отклонение (RMSE): 0.087130
[valve_0009_lidar_classes.ply] Сборка завершена  Сегментов: 6 Отклонение (RMSE): 0.061266
[valve_0010_lidar_cla